# Flaky Test Analyzer — Full Pipeline

Run all cells top to bottom.

| | Step | Output |
|-|------|--------|
| **1** | Generate 100 sample TRX files | `data/sample/` |
| **2** | Build HTML dashboard | `docs/index.html` + KPI summary |
| **3** | Suite overview table | inline |
| **4** | Per-runner failures table | inline |
| **5** | Charts: classification · failures · top flaky | inline |
| **6** | Full dashboard preview | IFrame inline |

> **Real CI data:** skip cell 1, set `DATA_DIR = Path('data/runs')` in the setup cell.

In [ ]:
# Shared setup — run once
import sys, os, json, importlib, warnings
from pathlib import Path

os.chdir('..')              # work from project root
sys.path.insert(0, 'src')  # expose src/ modules directly
warnings.filterwarnings('ignore')

import pandas as pd
import plotly.io as pio
import plotly.graph_objects as go
from plotly.subplots import make_subplots
pio.renderers.default = 'notebook'

DATA_DIR = Path('data/sample')  # ← change to data/runs for real CI
print('Setup complete')

## Step 1 — Generate Sample Data
_Skip if using real CI data._

In [ ]:
from generate_sample_data import main as gen
gen()

## Step 2 — Build Dashboard

In [ ]:
import build_dashboard as bd
importlib.reload(bd)

data = bd.compute(DATA_DIR)
html = bd.build_html(data)
Path('docs').mkdir(exist_ok=True)
Path('docs/index.html').write_text(html)
Path('data/dashboard.json').write_text(json.dumps(data, indent=2))

k = data['kpis']
for label, val in [
    ('Runs',       k['runs']),
    ('Tests/run',  k['tests_per_run']),
    ('Avg fails',  k['avg_fails']),
    ('Pass rate',  k['pass_rate']),
    ('Flaky',      k['flaky']),
    ('Broken',     k['broken']),
]:
    print(f'  {label:<12} {val}')
print('\n✓  docs/index.html written')

## Step 3 — Suite Overview Table

In [ ]:
from trx_parser import load_all

df = pd.DataFrame([{
    'run_id':      r.run_id,
    'test_name':   r.test_name,
    'outcome':     r.outcome,
    'duration_ms': r.duration_ms,
} for r in load_all(DATA_DIR)])

df['failed'] = (df['outcome'] == 'Failed').astype(int)
run_stats = (
    df.groupby('run_id')
      .agg(tests=('test_name', 'count'), fails=('failed', 'sum'))
      .reset_index()
)
print(f'Loaded  {df["run_id"].nunique()} runs  ·  {df["test_name"].nunique()} tests  ·  {len(df):,} rows')

summary = pd.DataFrame({
    'Metric': ['Runs checked','Tests per run','Avg fails / run',
               'Min fails','Max fails','Total failures','Pass rate'],
    'Value': [
        run_stats.shape[0],
        int(run_stats['tests'].mean()),
        round(run_stats['fails'].mean(), 1),
        int(run_stats['fails'].min()),
        int(run_stats['fails'].max()),
        int(run_stats['fails'].sum()),
        f"{(1 - df['failed'].sum()/len(df)):.2%}",
    ]
})

go.Figure(data=[go.Table(
    columnwidth=[220, 130],
    header=dict(
        values=['<b>Metric</b>', '<b>Value</b>'],
        fill_color='#1e293b', font=dict(color='white', size=13),
        align='left', height=38,
    ),
    cells=dict(
        values=[summary['Metric'], summary['Value']],
        fill_color=[['#f8fafc','white']*4, ['#f8fafc','white']*4],
        align='left', font=dict(size=13), height=34,
    )
)]).update_layout(margin=dict(l=0,r=0,t=10,b=0), height=290, width=440)

## Step 4 — Per-Runner Failures Table

In [ ]:
runner_tbl = (
    df[df['failed'] == 1]
    .groupby('run_id')['test_name']
    .agg(Fails='count', tests=list)
    .reset_index()
    .rename(columns={'run_id': 'Runner'})
    .sort_values('Fails', ascending=False)
    .reset_index(drop=True)
)
runner_tbl['Failing Tests'] = runner_tbl['tests'].apply(
    lambda x: '<br>'.join(sorted(set(x)))
)
runner_tbl = runner_tbl[['Runner', 'Fails', 'Failing Tests']]

max_f = runner_tbl['Fails'].max()
fill  = ['#fee2e2' if v/max_f >= .85 else '#fff7ed' if v/max_f >= .55 else '#f0fdf4'
          for v in runner_tbl['Fails']]

go.Figure(data=[go.Table(
    columnwidth=[110, 55, 640],
    header=dict(
        values=['<b>Runner</b>', '<b>Fails</b>', '<b>Failing Tests</b>'],
        fill_color='#1e293b', font=dict(color='white', size=12),
        align='left', height=36,
    ),
    cells=dict(
        values=[runner_tbl['Runner'], runner_tbl['Fails'], runner_tbl['Failing Tests']],
        fill_color=[['#f8fafc'] * len(runner_tbl), fill, ['white'] * len(runner_tbl)],
        align=['left', 'center', 'left'], font=dict(size=11),
    )
)]).update_layout(margin=dict(l=0,r=0,t=10,b=0), height=520)

## Step 5 — Charts

In [ ]:
test_rate = (
    df.groupby('test_name')['failed']
    .agg(['sum', 'count'])
    .assign(rate=lambda d: d['sum'] / d['count'])
)
flaky  = test_rate[(test_rate['rate'] >= 0.05) & (test_rate['rate'] < 0.8)]
broken = test_rate[test_rate['rate'] >= 0.8]
stable = test_rate[test_rate['rate'] < 0.05]

rs    = run_stats.sort_values('run_id').reset_index(drop=True)
top10 = flaky.sort_values('rate').tail(10)
short = [n.rsplit('_', 1)[0][-28:] for n in top10.index]

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=['Test classification', 'Failures per run', 'Top 10 flaky tests'],
    specs=[[{'type': 'pie'}, {'type': 'xy'}, {'type': 'xy'}]],
    horizontal_spacing=0.10,
)

fig.add_trace(go.Pie(
    labels=[f'Stable ({len(stable)})', f'Flaky ({len(flaky)})', f'Broken ({len(broken)})'],
    values=[len(stable), len(flaky), len(broken)],
    marker_colors=['#22c55e', '#f97316', '#ef4444'],
    hole=0.48, textinfo='label+percent', textfont=dict(size=11),
), row=1, col=1)

bar_c = ['#ef4444' if v >= 12 else '#f59e0b' if v >= 8 else '#22c55e' for v in rs['fails']]
fig.add_trace(go.Bar(
    x=list(range(len(rs))), y=rs['fails'], marker_color=bar_c, showlegend=False
), row=1, col=2)
avg_f = rs['fails'].mean()
fig.add_shape(type='line', x0=0, x1=len(rs)-1, y0=avg_f, y1=avg_f,
              line=dict(color='#334155', width=1.5, dash='dash'), row=1, col=2)
fig.add_annotation(x=len(rs)-1, y=avg_f, text=f' avg {avg_f:.1f}',
                   showarrow=False, font=dict(size=10, color='#64748b'), row=1, col=2)

fig.add_trace(go.Bar(
    x=top10['rate'] * 100, y=short, orientation='h',
    marker_color='#f97316', showlegend=False,
    text=[f'{v:.0f}%' for v in top10['rate'] * 100],
    textposition='auto', cliponaxis=False,
), row=1, col=3)

fig.update_layout(
    height=480, width=1150,
    title_text=f"Suite: {df['run_id'].nunique()} runs · {df['test_name'].nunique()} tests",
    title_font_size=13,
    plot_bgcolor='white', paper_bgcolor='white',
    margin=dict(t=55, b=30, l=20, r=80),
)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)
fig


## Step 6 — Full Dashboard Preview

In [ ]:
from IPython.display import IFrame
IFrame(src='docs/index.html', width='100%', height=920)